# Regime-Aware Cross-Asset Market Risk

Everything is kept in this single notebook: configuration, data loading, risk metrics, regime detection, backtesting, attribution, stress testing, plots, and interpretation.

## 1. Imports and configuration

In [ ]:
from collections import OrderedDict
from collections.abc import Mapping, Sequence
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from scipy.optimize import minimize
from scipy.stats import chi2, norm
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

WEIGHTS = OrderedDict({"SPY":0.20,"TLT":0.20,"LQD":0.20,"GLD":0.20,"FXB":0.20})
START_DATE = "2015-01-01"
VAR_WINDOW = 250
REGIME_WINDOW = 60
EWMA_LAMBDA = 0.94

## 2. Data loading and portfolio construction

In [ ]:
def download_adjusted_close(tickers: Sequence[str], start: str, end: str | None = None) -> pd.DataFrame:
    raw = yf.download(list(tickers), start=start, end=end, auto_adjust=True, progress=False, group_by="column", threads=True)
    if raw.empty: raise ValueError("No market data returned.")
    prices = raw["Close"].copy() if isinstance(raw.columns, pd.MultiIndex) else raw[["Close"]].copy()
    if not isinstance(raw.columns, pd.MultiIndex): prices.columns = [str(tickers[0])]
    prices = prices.reindex(columns=list(tickers)).dropna(how="any").sort_index()
    if prices.empty: raise ValueError("No overlapping observations remain.")
    if (prices <= 0).any().any(): raise ValueError("Non-positive prices detected.")
    return prices

def simple_returns(prices: pd.DataFrame) -> pd.DataFrame:
    r = prices.pct_change(fill_method=None).dropna(how="any")
    if r.empty: raise ValueError("Not enough observations to calculate returns.")
    return r

def validate_weights(weights: Mapping[str,float], columns: pd.Index) -> pd.Series:
    w = pd.Series(weights, dtype=float)
    if set(w.index) != set(columns): raise ValueError("Weights and assets do not match.")
    if not np.isclose(w.sum(),1.0): raise ValueError("Weights must sum to 1.")
    return w.reindex(columns)

def portfolio_returns(asset_returns: pd.DataFrame, weights: Mapping[str,float]) -> pd.Series:
    w = validate_weights(weights, asset_returns.columns)
    out = asset_returns.mul(w, axis=1).sum(axis=1)
    out.name = "portfolio_return"
    return out

## 3. Core risk metrics

In [ ]:
def historical_var(returns, confidence=0.99):
    s = pd.Series(returns,dtype=float).dropna()
    return max(0.0, -float(s.quantile(1-confidence)))

def historical_expected_shortfall(returns, confidence=0.99):
    s = pd.Series(returns,dtype=float).dropna()
    q = float(s.quantile(1-confidence))
    return max(0.0, -float(s[s <= q].mean()))

def parametric_var(returns, confidence=0.99):
    s = pd.Series(returns,dtype=float).dropna()
    q = float(s.mean()) + float(norm.ppf(1-confidence))*float(s.std(ddof=1))
    return max(0.0, -q)

def ewma_variance(returns, lam=EWMA_LAMBDA):
    s = pd.Series(returns,dtype=float).dropna()
    n0=min(20,len(s)); v=float(s.iloc[:n0].var(ddof=1))
    for r in s.iloc[n0:]: v = lam*v + (1-lam)*float(r)**2
    return v

def ewma_var(returns, confidence=0.99, lam=EWMA_LAMBDA):
    return float(norm.ppf(confidence))*np.sqrt(ewma_variance(returns,lam))

## 4. Regime features: volatility, correlation and PCA

In [ ]:
def average_pairwise_correlation(x):
    corr=x.corr().to_numpy(dtype=float); n=corr.shape[0]
    return float(np.nanmean(corr[~np.eye(n,dtype=bool)]))

def first_pc_variance_share(x):
    scaled=StandardScaler().fit_transform(x.dropna(how="any"))
    return float(PCA(n_components=1).fit(scaled).explained_variance_ratio_[0])

def rolling_regime_features(asset_returns, port_returns, window=REGIME_WINDOW, annualisation=252):
    rows=[]
    for end in range(window,len(asset_returns)+1):
        aw=asset_returns.iloc[end-window:end]; pw=port_returns.iloc[end-window:end]
        rows.append({"date":asset_returns.index[end-1],
                     "portfolio_volatility":float(pw.std(ddof=1)*np.sqrt(annualisation)),
                     "average_correlation":average_pairwise_correlation(aw),
                     "pc1_variance_share":first_pc_variance_share(aw)})
    return pd.DataFrame(rows).set_index("date")

def expanding_percentile_score(features, min_periods=60):
    cols=["portfolio_volatility","average_correlation","pc1_variance_share"]
    score=pd.Series(index=features.index,dtype=float,name="regime_stress_score")
    for i in range(len(features)):
        h=features.iloc[:i+1]
        if len(h)<min_periods: continue
        current=h.iloc[-1]
        score.iloc[i]=np.mean([(h[c] <= current[c]).mean() for c in cols])
    return score

def label_regimes(score):
    clean=score.dropna(); labels=pd.Series(index=score.index,dtype="object",name="regime")
    if clean.empty: return labels
    low,high=clean.quantile([1/3,2/3])
    labels.loc[score<=low]="calm"
    labels.loc[(score>low)&(score<=high)]="transitional"
    labels.loc[score>high]="stressed"
    return labels

## 5. Walk-forward VaR backtesting

In [ ]:
def rolling_historical_var(returns, window=VAR_WINDOW, confidence=0.99):
    s=pd.Series(returns,dtype=float).dropna(); f=pd.Series(index=s.index,dtype=float,name="historical_var")
    for i in range(window,len(s)): f.iloc[i]=historical_var(s.iloc[i-window:i],confidence)
    return f

def rolling_ewma_var(returns, confidence=0.99, lam=EWMA_LAMBDA, min_periods=60):
    s=pd.Series(returns,dtype=float).dropna(); f=pd.Series(index=s.index,dtype=float,name="ewma_var")
    v=float(s.iloc[:min_periods].var(ddof=1)); z=float(norm.ppf(confidence))
    for i in range(min_periods,len(s)):
        v=lam*v+(1-lam)*float(s.iloc[i-1])**2
        f.iloc[i]=z*np.sqrt(max(v,0.0))
    return f

def var_exceptions(returns,var_forecasts):
    a=pd.concat([pd.Series(returns,dtype=float).rename("return"),var_forecasts.rename("var")],axis=1).dropna()
    return (a["return"] < -a["var"]).rename("exception")

def kupiec_pof_test(exceptions, confidence=0.99):
    obs=pd.Series(exceptions).dropna().astype(bool); n=len(obs); x=int(obs.sum()); p=1-confidence; phat=x/n
    eps=np.finfo(float).eps; p0=np.clip(p,eps,1-eps); p1=np.clip(phat,eps,1-eps)
    ll0=(n-x)*np.log(1-p0)+x*np.log(p0); ll1=(n-x)*np.log(1-p1)+x*np.log(p1)
    lr=float(-2*(ll0-ll1))
    return {"observations":n,"exceptions":x,"expected_exception_rate":p,"observed_exception_rate":phat,"lr_stat":lr,"p_value":float(chi2.sf(lr,1))}

## 6. Risk attribution and stress testing

In [ ]:
def component_volatility_contributions(covariance, weights):
    w=validate_weights(weights,covariance.columns); sigma=covariance.to_numpy(float); wv=w.to_numpy(float)
    vol=np.sqrt(float(wv @ sigma @ wv)); marginal=sigma @ wv / vol; component=wv*marginal
    return pd.DataFrame({"weight":wv,"component_volatility":component,"pct_of_portfolio_volatility":component/vol},index=covariance.columns)

def scenario_portfolio_return(shocks, weights):
    s=pd.Series(shocks,dtype=float); w=validate_weights(weights,s.index); return float(w @ s)

def historical_worst_window(port_returns, horizon=5):
    s=pd.Series(port_returns,dtype=float).dropna(); compounded=(1+s).rolling(horizon).apply(np.prod,raw=True)-1
    end=compounded.idxmin(); loc=s.index.get_loc(end); start=s.index[loc-horizon+1]
    return {"start_date":start,"end_date":end,"horizon_days":horizon,"compounded_return":float(compounded.loc[end])}

def reverse_stress_test(covariance, weights, target_loss=0.05, bounds=None):
    w=validate_weights(weights,covariance.columns); sigma=covariance.to_numpy(float); wv=w.to_numpy(float)
    pv=float(wv @ sigma @ wv); analytical=-(target_loss/pv)*(sigma @ wv)
    bounds=[(-0.30,0.30)]*len(w) if bounds is None else list(bounds)
    if all(lo<=x<=hi for x,(lo,hi) in zip(analytical,bounds)):
        return pd.Series(analytical,index=covariance.columns,name="reverse_stress_shock")
    precision=np.linalg.pinv(sigma)
    objective=lambda x: float(0.5*x @ precision @ x)
    constraint=lambda x: float(-target_loss-wv @ x)
    x0=np.array([np.clip(x,lo,hi) for x,(lo,hi) in zip(analytical,bounds)],dtype=float)
    if constraint(x0)<0: x0=np.array([lo for lo,_ in bounds],dtype=float)
    result=minimize(objective,x0=x0,method="SLSQP",bounds=bounds,constraints={"type":"ineq","fun":constraint},options={"ftol":1e-10,"maxiter":2000})
    if not result.success: raise RuntimeError(result.message)
    return pd.Series(result.x,index=covariance.columns,name="reverse_stress_shock")

# Analysis

## 7. Download prices and build the portfolio

In [ ]:
tickers=list(WEIGHTS)
prices=download_adjusted_close(tickers,START_DATE)
asset_returns=simple_returns(prices)
port_ret=portfolio_returns(asset_returns,WEIGHTS)
display(prices.tail())
display(port_ret.describe().to_frame("portfolio_return"))

## 8. Baseline VaR and Expected Shortfall

In [ ]:
risk_summary=pd.DataFrame({
    "95%":{"Historical VaR":historical_var(port_ret,.95),"Parametric VaR":parametric_var(port_ret,.95),"Historical ES":historical_expected_shortfall(port_ret,.95)},
    "99%":{"Historical VaR":historical_var(port_ret,.99),"Parametric VaR":parametric_var(port_ret,.99),"Historical ES":historical_expected_shortfall(port_ret,.99)}
})
risk_summary

## 9. Regime analysis

In [ ]:
features=rolling_regime_features(asset_returns,port_ret)
stress_score=expanding_percentile_score(features)
regimes=label_regimes(stress_score)
regime_frame=features.join(stress_score).join(regimes)
display(regime_frame.tail())

for col,title in [("portfolio_volatility","Rolling Annualised Portfolio Volatility"),("average_correlation","Rolling Average Cross-Asset Correlation"),("pc1_variance_share","Rolling PC1 Variance Share"),("regime_stress_score","Regime Stress Score")]:
    regime_frame[col].plot(figsize=(12,4),title=title); plt.show()

## 10. Walk-forward VaR backtest

In [ ]:
hist_var_99=rolling_historical_var(port_ret,VAR_WINDOW,.99)
ewma_var_99=rolling_ewma_var(port_ret,.99,EWMA_LAMBDA)
hist_exceptions=var_exceptions(port_ret,hist_var_99)
ewma_exceptions=var_exceptions(port_ret,ewma_var_99)

kupiec_results=pd.DataFrame({"Historical VaR":kupiec_pof_test(hist_exceptions,.99),"EWMA VaR":kupiec_pof_test(ewma_exceptions,.99)})
display(kupiec_results)

plot_frame=pd.concat([port_ret,hist_var_99,ewma_var_99],axis=1).dropna()
ax=plot_frame["portfolio_return"].plot(figsize=(12,5),title="Portfolio Returns vs 99% VaR")
(-plot_frame["historical_var"]).plot(ax=ax,label="Historical VaR threshold")
(-plot_frame["ewma_var"]).plot(ax=ax,label="EWMA VaR threshold")
plt.legend(); plt.show()

## 11. VaR breaches by regime

In [ ]:
breach_frame=pd.DataFrame({"historical_exception":hist_exceptions,"ewma_exception":ewma_exceptions}).join(regime_frame[["regime_stress_score","regime"]],how="left")
breach_by_regime=breach_frame.dropna(subset=["regime"]).groupby("regime")[["historical_exception","ewma_exception"]].mean()
breach_by_regime

## 12. Risk attribution and reverse stress test

In [ ]:
recent_cov=asset_returns.tail(VAR_WINDOW).cov()
contributions=component_volatility_contributions(recent_cov,WEIGHTS)
reverse_shock=reverse_stress_test(recent_cov,WEIGHTS,target_loss=.05)
worst_5_day=historical_worst_window(port_ret,5)

display(recent_cov)
display(contributions)
display(reverse_shock.to_frame())
print("Portfolio return under reverse-stress shock:",f"{scenario_portfolio_return(reverse_shock.to_dict(),WEIGHTS):.2%}")
print("Worst historical five-day window:",worst_5_day)

## 13. Interpretation

Use the outputs above to compare Historical vs EWMA VaR, check whether breaches are concentrated in stressed regimes, identify which asset drives current volatility, and compare the reverse-stress scenario with the worst historical five-day loss.

**Limitation:** the stress score is walk-forward, but the final calm/transitional/stressed tertile cutoffs use the full score history, so the labels are descriptive rather than fully out-of-sample.